In [ ]:
import pandas as pd
import os

# Define the base path to the directory
base_path = '/content/drive/MyDrive/Colab Notebooks/Datasets/Anonymized Vessle AIS data'

# Assuming the trawler data is in a CSV file named 'trawler_data.csv' within that directory
file_name = 'trawlers.csv'
file_path = os.path.join(base_path, file_name)

# Load the CSV file into a pandas DataFrame
try:
    trawler_df = pd.read_csv(file_path)
    print(f"Successfully loaded data from {file_path}")
    # Display the first 5 rows of the DataFrame
    display(trawler_df.head())
except FileNotFoundError:
    print(f"Error: The file '{file_name}' was not found at '{base_path}'. Please check the path and filename.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully loaded data from /content/drive/MyDrive/Colab Notebooks/Datasets/Anonymized Vessle AIS data/trawlers.csv


,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source
0,1.252340e+12,1.325376e+09,0.0,0.0,0.0,153.0,52.458649,4.581200,-1.0,gfw
1,1.252340e+12,1.325378e+09,0.0,0.0,0.0,153.0,52.458668,4.581167,-1.0,gfw
2,1.252340e+12,1.325379e+09,0.0,0.0,0.0,153.0,52.458633,4.581183,-1.0,gfw
3,1.252340e+12,1.325380e+09,0.0,0.0,0.0,153.0,52.458649,4.581234,-1.0,gfw
4,1.252340e+12,1.325381e+09,0.0,0.0,0.0,153.0,52.458649,4.581183,-1.0,gfw


In [ ]:
trawler_df.shape

(4369023, 10)

Removal of Port Static

In [ ]:
# Drop rows where distance_from_port < 3000 and speed < 0.2
# This is equivalent to keeping rows where NOT (distance_from_port < 3000 and speed < 0.2)
filtered_trawler_df = trawler_df[~((trawler_df['distance_from_port'] < 3000) & (trawler_df['speed'] < 0.2))]

# Display the shape and preview
print(f"Original shape: {trawler_df.shape}")
print(f"Filtered shape: {filtered_trawler_df.shape}")
display(filtered_trawler_df.head())

Original shape: (4369023, 10)
Filtered shape: (3175234, 10)


,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source
38,1.252340e+12,1.325413e+09,0.0000,0.0,102.300003,360.0,52.458767,4.581183,-1.0,gfw
242,1.252340e+12,1.325682e+09,826520.6875,855189.5,0.000000,33.0,-13.951197,4.580761,-1.0,gfw
345,1.252340e+12,1.325780e+09,0.0000,0.0,102.300003,360.0,52.458599,4.581250,-1.0,gfw
633,1.252340e+12,1.326075e+09,0.0000,0.0,102.300003,360.0,52.458569,4.581300,-1.0,gfw
784,1.252340e+12,1.327070e+09,826520.6875,855189.5,0.000000,194.0,-13.951197,4.580761,-1.0,gfw


Drop Original dataframe

In [ ]:
import gc

# Delete the original DataFrame from memory
try:
    del trawler_df
    print("Successfully deleted trawler_df reference.")
except NameError:
    print("trawler_df is already deleted or does not exist.")

# Force garbage collection to reclaim the memory
gc.collect()

Successfully deleted trawler_df reference.


17

#VOYAGE SEGEMTATION

In [ ]:
import numpy as np

# Ensure sorted order
filtered_trawler_df = filtered_trawler_df.sort_values(by=['mmsi', 'timestamp']).reset_index(drop=True)

# Calculate time gap within each mmsi group
gaps = filtered_trawler_df.groupby('mmsi')['timestamp'].diff()

# A new voyage starts if it's the first record of that mmsi (NaN gap) or if the gap exceeds 7200 seconds
is_new_voyage = gaps.isna() | (gaps > 7200)

# Generate a sequential voyage counter starting at 1 for each mmsi
filtered_trawler_df['voyage_seq'] = filtered_trawler_df.groupby('mmsi').cumcount() # fallback placeholder
# Compute cumsum of the new voyage indicators to get the sequence number per mmsi
filtered_trawler_df['voyage_seq'] = filtered_trawler_df.groupby('mmsi')['timestamp'].transform(
    lambda x: (x.diff() > 7200).cumsum() + 1
)

# Create the final voyage_id string in the format: mmsi_voyage_sequence
# Using astype(str) to avoid float formatting for mmsi
filtered_trawler_df['voyage_id'] = filtered_trawler_df['mmsi'].astype(int).astype(str) + '_' + filtered_trawler_df['voyage_seq'].astype(str)

# Drop the intermediate helper column
filtered_trawler_df = filtered_trawler_df.drop(columns=['voyage_seq'])

# Display the top rows to verify the newly added column
display(filtered_trawler_df[['mmsi', 'timestamp', 'dt', 'voyage_id']].head(58918))

,mmsi,timestamp,dt,voyage_id
0,1.252340e+12,1.325413e+09,NaN,1252339803566_1
1,1.252340e+12,1.325682e+09,269397.0,1252339803566_2
2,1.252340e+12,1.325780e+09,97471.0,1252339803566_3
3,1.252340e+12,1.326075e+09,295162.0,1252339803566_4
4,1.252340e+12,1.327070e+09,995066.0,1252339803566_5
...,...,...,...,...
58913,1.252340e+12,1.480016e+09,9981.0,1252339803566_1709
58914,1.252340e+12,1.480021e+09,5358.0,1252339803566_1709
58915,5.020143e+12,1.373578e+09,NaN,5020143137211_1
58916,5.020143e+12,1.373582e+09,4242.0,5020143137211_1


#FEATURE ENGINEERING

**dt**

In [ ]:
# Calculate the timestep difference (dt) grouped by each unique mmsi
filtered_trawler_df['dt'] = filtered_trawler_df.groupby('mmsi')['timestamp'].diff()

# Display the head to verify, including checking some rows
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0
...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0


In [ ]:
filtered_trawler_df['source'].unique()

array(['gfw', 'false_positives', 'dalhousie_trawl'], dtype=object)

**log dt**

In [ ]:
import numpy as np

# Compute log(1 + dt) to handle any potential 0-second transitions smoothly
filtered_trawler_df['log_dt'] = np.log1p(filtered_trawler_df['dt'])

# Display the updated DataFrame to verify the transformation
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN,NaN
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0,11.487320
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0,12.595283
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565
...,...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0,9.208539
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0,8.586533
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN,NaN
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0,8.353026


**log distances**

In [ ]:
# Create log features for distance_from_shore and distance_from_port
filtered_trawler_df['log_distance_from_shore'] = np.log1p(filtered_trawler_df['distance_from_shore'])
filtered_trawler_df['log_distance_from_port'] = np.log1p(filtered_trawler_df['distance_from_port'])

# Display the top 58918 rows to verify
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt,log_distance_from_shore,log_distance_from_port
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN,NaN,0.000000,0.000000
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945,13.624981,13.659080
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0,11.487320,0.000000,0.000000
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0,12.595283,0.000000,0.000000
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565,13.624981,13.659080
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0,9.208539,12.619415,12.794215
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0,8.586533,12.654019,12.812742
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN,NaN,0.000000,6.908730
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0,8.353026,6.908730,6.908730


**d_course**

In [ ]:
# Calculate course difference (d_course) grouped by mmsi
# Formula: ((course - prev_course + 180) % 360) - 180
prev_course = filtered_trawler_df.groupby('mmsi')['course'].shift(1)
filtered_trawler_df['d_course'] = ((filtered_trawler_df['course'] - prev_course + 180) % 360) - 180

# Display the top 58918 rows to verify
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt,log_distance_from_shore,log_distance_from_port,d_course
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN,NaN,0.000000,0.000000,NaN
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945,13.624981,13.659080,33.000000
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0,11.487320,0.000000,0.000000,-33.000000
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0,12.595283,0.000000,0.000000,0.000000
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565,13.624981,13.659080,-166.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0,9.208539,12.619415,12.794215,37.000000
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0,8.586533,12.654019,12.812742,78.000000
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN,NaN,0.000000,6.908730,NaN
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0,8.353026,6.908730,6.908730,-171.500008


**dx and dy from lat and lon using conversion formulae**

In [ ]:
# Calculate previous lat and lon grouped by mmsi
prev_lat = filtered_trawler_df.groupby('mmsi')['lat'].shift(1)
prev_lon = filtered_trawler_df.groupby('mmsi')['lon'].shift(1)

# Compute dx and dy according to the provided formulae
filtered_trawler_df['dx'] = (filtered_trawler_df['lon'] - prev_lon) * np.cos(np.radians((prev_lat + filtered_trawler_df['lat']) / 2)) * 111.32
filtered_trawler_df['dy'] = (filtered_trawler_df['lat'] - prev_lat) * 110.57

# Display the top 58918 rows to verify the changes
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt,log_distance_from_shore,log_distance_from_port,d_course,dx,dy
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945,13.624981,13.659080,33.000000,-0.044350,-7342.949676
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0,11.487320,0.000000,0.000000,-33.000000,0.051365,7342.931117
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0,12.595283,0.000000,0.000000,0.000000,0.003364,-0.003374
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565,13.624981,13.659080,-166.000000,-0.056577,-7342.927743
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0,9.208539,12.619415,12.794215,37.000000,8.973172,1.997603
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0,8.586533,12.654019,12.812742,78.000000,-8.741217,-0.254340
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN,NaN,0.000000,6.908730,NaN,NaN,NaN
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0,8.353026,6.908730,6.908730,-171.500008,0.765175,0.614550


**Velocity vectors vx vy**

In [ ]:
# Calculate velocity vectors vx and vy
filtered_trawler_df['vx'] = filtered_trawler_df['speed'] * np.sin(filtered_trawler_df['course'] * np.pi / 180)
filtered_trawler_df['vy'] = filtered_trawler_df['speed'] * np.cos(filtered_trawler_df['course'] * np.pi / 180)

# Display the top 58918 rows to verify
display(filtered_trawler_df.head(58918))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt,log_distance_from_shore,log_distance_from_port,d_course,dx,dy,vx,vy
0,1.252340e+12,1.325413e+09,0.000000,0.000000,102.300003,360.000000,52.458767,4.581183,-1.0,gfw,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,-2.505627e-14,102.300003
1,1.252340e+12,1.325682e+09,826520.687500,855189.500000,0.000000,33.000000,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945,13.624981,13.659080,33.000000,-0.044350,-7342.949676,0.000000e+00,0.000000
2,1.252340e+12,1.325780e+09,0.000000,0.000000,102.300003,360.000000,52.458599,4.581250,-1.0,gfw,97471.0,11.487320,0.000000,0.000000,-33.000000,0.051365,7342.931117,-2.505627e-14,102.300003
3,1.252340e+12,1.326075e+09,0.000000,0.000000,102.300003,360.000000,52.458569,4.581300,-1.0,gfw,295162.0,12.595283,0.000000,0.000000,0.000000,0.003364,-0.003374,-2.505627e-14,102.300003
4,1.252340e+12,1.327070e+09,826520.687500,855189.500000,0.000000,194.000000,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565,13.624981,13.659080,-166.000000,-0.056577,-7342.927743,-0.000000e+00,-0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58913,1.252340e+12,1.480016e+09,302371.500000,360127.250000,3.700000,257.000000,61.926468,-2.435533,-1.0,gfw,9981.0,9.208539,12.619415,12.794215,37.000000,8.973172,1.997603,-3.605169e+00,-0.832319
58914,1.252340e+12,1.480021e+09,313017.875000,366861.531250,2.100000,335.000000,61.924168,-2.602383,-1.0,gfw,5358.0,8.586533,12.654019,12.812742,78.000000,-8.741217,-0.254340,-8.874983e-01,1.903246
58915,5.020143e+12,1.373578e+09,0.000000,999.975464,1.300000,210.600006,50.715828,-127.487816,-1.0,false_positives,NaN,NaN,0.000000,6.908730,NaN,NaN,NaN,-6.617539e-01,-1.118965
58916,5.020143e+12,1.373582e+09,999.975464,999.975464,5.500000,39.099998,50.721386,-127.476959,-1.0,false_positives,4242.0,8.353026,6.908730,6.908730,-171.500008,0.765175,0.614550,3.468717e+00,4.268255


#FEATURE ENGINEERING DONE

In [ ]:
# Filter the DataFrame for the specific voyage and count the number of records
voyage_points = filtered_trawler_df[filtered_trawler_df['voyage_id'] == '274850145767759_1916']
print(f"Number of points in voyage '274850145767759_1905 ': {len(voyage_points)}")

Number of points in voyage '274850145767759_1905 ': 13214


In [ ]:
display(filtered_trawler_df.head(5))

,mmsi,timestamp,distance_from_shore,distance_from_port,speed,course,lat,lon,is_fishing,source,dt,log_dt,log_distance_from_shore,log_distance_from_port,d_course,dx,dy,vx,vy,voyage_id
0,1.252340e+12,1.325413e+09,0.0000,0.0,102.300003,360.0,52.458767,4.581183,-1.0,gfw,NaN,NaN,0.000000,0.00000,NaN,NaN,NaN,-2.505627e-14,102.300003,1252339803566_1
1,1.252340e+12,1.325682e+09,826520.6875,855189.5,0.000000,33.0,-13.951197,4.580761,-1.0,gfw,269397.0,12.503945,13.624981,13.65908,33.0,-0.044350,-7342.949676,0.000000e+00,0.000000,1252339803566_2
2,1.252340e+12,1.325780e+09,0.0000,0.0,102.300003,360.0,52.458599,4.581250,-1.0,gfw,97471.0,11.487320,0.000000,0.00000,-33.0,0.051365,7342.931117,-2.505627e-14,102.300003,1252339803566_3
3,1.252340e+12,1.326075e+09,0.0000,0.0,102.300003,360.0,52.458569,4.581300,-1.0,gfw,295162.0,12.595283,0.000000,0.00000,0.0,0.003364,-0.003374,-2.505627e-14,102.300003,1252339803566_4
4,1.252340e+12,1.327070e+09,826520.6875,855189.5,0.000000,194.0,-13.951197,4.580761,-1.0,gfw,995066.0,13.810565,13.624981,13.65908,-166.0,-0.056577,-7342.927743,-0.000000e+00,-0.000000,1252339803566_5


In [ ]:
import io

# Calculate approximate CSV size in memory
buf = io.BytesIO()
filtered_trawler_df.to_csv(buf, index=False)
csv_size_bytes = buf.tell()
csv_size_mb = csv_size_bytes / (1024 * 1024)
print(f"Expected CSV file size: {csv_size_mb:.2f} MB ({csv_size_bytes:,} bytes)")

Expected CSV file size: 855.32 MB (896,866,406 bytes)


In [ ]:
import io

# Calculate approximate Parquet size in memory using pyarrow engine
buf_parquet = io.BytesIO()
filtered_trawler_df.to_parquet(buf_parquet, index=False, engine='pyarrow')
parquet_size_bytes = buf_parquet.tell()
parquet_size_mb = parquet_size_bytes / (1024 * 1024)

print(f"Expected Parquet file size: {parquet_size_mb:.2f} MB ({parquet_size_bytes:,} bytes)")
print(f"Compression ratio (Parquet vs CSV): {csv_size_bytes / parquet_size_bytes:.2f}x smaller")

Expected Parquet file size: 158.67 MB (166,377,717 bytes)
Compression ratio (Parquet vs CSV): 5.39x smaller


In [ ]:
import os

# Define output path
output_filename = 'filtered_trawlers.parquet'
output_filepath = os.path.join(base_path, output_filename)

# Save the DataFrame as a Parquet file
try:
    filtered_trawler_df.to_parquet(output_filepath, index=False, engine='pyarrow')
    print(f"Successfully saved the filtered DataFrame as Parquet to: {output_filepath}")
    print(f"File size on disk: {os.path.getsize(output_filepath) / (1024 * 1024):.2f} MB")
except Exception as e:
    print(f"An error occurred while saving: {e}")

Successfully saved the filtered DataFrame as Parquet to: /content/drive/MyDrive/Colab Notebooks/Datasets/Anonymized Vessle AIS data/filtered_trawlers.parquet
File size on disk: 158.67 MB
